# 自动微分

上一章中，我们使用微积分的链式法则，分别计算了推理函数和损失函数的偏导数，并由此获得了模型参数的梯度。在现实应用中，网络模型可能有几十层、包含成千上万不同的人工神经元。位于网络不同位置的神经元，参数的梯度计算公式也各不相同。因此我们需要构建一种机制，可以自动完成每个基本运算的求导，并根据各个参数在网络中的位置进行链式合并。

在深度学习中采用的方法叫做**计算图**。

## 计算图

神经网络的计算并不是一步到位的。输入数据经过一层一层的神经元，每一步只做一个简单的局部运算，最终汇合到损失函数，得到损失值。这条数据流动的路径，就是**前向传播**。

为了能够计算梯度，我们需要在前向传播过程中，把数据流动的整个**拓扑结构**全部记录下来：哪个节点的输出，流向了哪个节点的输入。这个记录下来的拓扑结构，就称为**计算图**（Computational Graph）。

计算图中的每个**节点**可能是一个特征值、模型参数、中间计算结果，或者最终的损失值。节点之间的**边**代表数据的流向和依赖关系。有了计算图，我们就能沿着它反向追溯，逐级计算每个参数的梯度。

因为计算图是在前向传播过程中**动态**构建的，所以称为**动态计算图**（Dynamic Computational Graph）。与之对应的是**静态计算图**（Static Computational Graph），先把整个图的结构定义好，再送入数据运行。PyTorch 采用动态计算图，TensorFlow 早期采用静态计算图（新版本也支持动态模式）。我们将采用动态计算图。

## 数据链路

围绕着计算图，深度学习构建了三条数据链路，基本涵盖了神经网络训练的全部过程：

* **前向传播链路**：从输入数据（特征值）经过网络参数的层层计算，推理出输出数据（预测值）的过程。也称为推理链路，同时也是计算图的构建链路。
* **反向传播链路**：从损失函数计算损失项开始，反向沿着计算图计算所有节点梯度的过程。也称为梯度计算链路。
* **参数更新链路**：遍历所有参数，根据反向传播链路计算的梯度更新参数数值。


In [1]:
import numpy as np

## 张量

为了实现计算图，我们首先需要把每个节点从一个简单的数值（或者数组）扩展成一个可以容纳更多信息的结构，称为**张量**（Tensor）。

在数学上，张量是标量、向量和矩阵的统称与推广：单独一个数是**标量**（0 维张量），一组数排成一行是**向量**（1 维张量），数排成行列是**矩阵**（2 维张量），更高维度的数组统称为**张量**。

在深度学习的框架里，张量是对多维数组的进一步**封装**。张量内部还包括构建计算图必要的数据结构。张量将是计算图和反向传播链路的主要载体。

### 数据

每个张量将包括 4 类数据：

* **data**（数值）：这是每个节点的基本信息，比如标签值、参数值、损失值等。
* **grad**（梯度）：这是根据链式法则，反向推导到这个节点的梯度值。
* **gradient_fn**（梯度函数）：一个闭包函数，封装了当前节点的链式法则计算逻辑。
* **parents**（父节点列表）：这是本节点的所有直接上游节点。梯度计算将沿着这个列表继续反向传播。

### 反向函数

* **backawrd()**（反向函数）：这是张量内部最重要的一个函数，是链式法则的执行机制。每个张量的反向函数首先调用本张量的梯度函数，然后递归调用所有父节点的反向函数。

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据

### 特征、标签

所有的输入数据（特征值、标签值）都被封装成张量。所有的输入数据在计算图中都是子节点。它们没有父节点，所以也不需要设置梯度函数。同时，我们也不需要计算输入数据的梯度。

In [3]:
feature = Tensor([28.1, 58.0])
label = Tensor([165])

## 模型

### 权重、偏置

所有的模型参数（权重、偏置）也都被封装成张量。所有的模型参数在计算图中同样都是子节点。它们也没有父节点，也不需要设置梯度函数。和输入数据不同的是，我们需要计算模型参数的梯度，进而更新模型参数的数值。

### 推理函数

推理函数的输出数据（预测值）也要封装成张量。所有的中间数据和输出数据都不是子节点，它们有父节点，自然就也需要设置梯度函数。

我们在输出数据 $p$ 的梯度函数 **gradient_fn** 中计算了权重和偏置的梯度。

我们实际上并没有设置 $p$ 的父节点列表 **parents**，因为它的所有父节点（特征值、权重和偏置）都是子节点，本身没有梯度函数，无需参与梯度计算链路。

由此可见，模型是前向传播链路的主要载体，同时也是计算图构建的主要环节。

---

模型不再需要反向函数，他的功能基本被预测值的梯度函数取代。

但是模型提供了一个新的参数列表 **parameters**，包括了需要更新数值的模型参数。这个参数列表将在参数更新链路中被使用。


In [4]:
class Linear:

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def __call__(self, x: Tensor):
        return self.forward(x)

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad * x.data
            self.bias.grad += np.sum(p.grad)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

## 损失函数（均方误差）

损失函数中，我们同样将损失值封装成了张量。损失值张量的父节点包括预测值 $p$，而它的梯度函数则计算了预测值 $p$ 的梯度，也就是误差项。

损失函数自身没有参数需要更新，因此无需提供参数列表 $parameters$。

损失函数是前向传播链路的终点，也是计算图构建的终点；同时，它也是反向传播链路的起点。通过调用损失值张量的反向函数 $backward()$，整张计算图将被反向递归调用，完成所有参数梯度的计算。

In [5]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

## 优化器（随机梯度下降）

In [6]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

In [7]:
LEARNING_RATE = 0.00001

In [8]:
layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

In [9]:
prediction = layer(feature)
loss = loss_fn(prediction, label)
loss.backward()
optimizer.step()

In [10]:
prediction = layer(feature)
loss = loss_fn(prediction, label)
print(f'prediction:\t{prediction}\nloss:\t{loss}')

prediction:	Tensor([53.18309379])
loss:	Tensor(12503.020514375934)
